[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-05-schema.ipynb)

---
# Day 5 · Schema Management and Data Contracts
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Production Readiness

> **Goal for today:** Understand dlt's automatic schema inference, declare explicit column hints, observe schema evolution when source data changes, and implement data contracts to control (or block) those changes in production.

---
## Why schema management matters

dlt's auto-inference is great for exploration, but production pipelines need guarantees:

| Scenario | What you want |
|---|---|
| New field appears in source | Add column automatically (evolution) **or** reject the row (contract) |
| Field changes type (int → string) | Coerce or raise an error |
| Critical field goes missing | Raise immediately, don't silently load NULLs |
| Rogue field added by a new engineer | Discard it silently **or** block the whole load |

### Schema vs data contracts

| | Schema | Data contract |
|---|---|---|
| **What it controls** | Column names, types, nullability, primary keys | What happens when schema changes (new column, type change) |
| **Where you set it** | `@dlt.resource` column hints | `schema_contract` on pipeline or resource |
| **When it applies** | At normalization time | At normalization time |

Think of **schema** as the blueprint and **data contracts** as the enforcement policy.

---
## The four contract modes

You can set a contract on `tables`, `columns`, and/or `data_type` independently:

| Mode | What happens when a new column appears |
|---|---|
| `"evolve"` | Column added automatically — the schema grows (**default**) |
| `"discard_value"` | Row is loaded but the unknown column's value is set to NULL |
| `"discard_row"` | The whole row is silently dropped |
| `"freeze"` | A `SchemaFrozenException` is raised — load aborted |

You can apply the same modes to `tables` (to control whether new tables can be created) and `data_type` (to control whether existing column types can change).

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · Explicit column hints with `@dlt.resource`

Use `columns=` to declare the exact schema you expect. dlt will use your hints instead of inferring from data.

In [ ]:
import dlt

@dlt.resource(
    name="orders",
    primary_key="order_id",
    write_disposition="replace",
    # columns= declares the expected schema explicitly.
    # data_type options: text, bigint, double, bool, date, timestamp, json, binary
    # nullable=False means dlt will raise if NULLs appear in that column.
    columns={
        "order_id":    {"data_type": "bigint",    "nullable": False, "primary_key": True},
        "customer_id": {"data_type": "bigint",    "nullable": False},
        "amount":      {"data_type": "double",    "nullable": False},
        "status":      {"data_type": "text",      "nullable": False},
        "note":        {"data_type": "text",      "nullable": True},   # optional field
        "created_at":  {"data_type": "timestamp", "nullable": False},
    },
)
def orders_v1():
    yield [
        {"order_id": 1, "customer_id": 10, "amount": 99.99,  "status": "pending",  "note": None,          "created_at": "2024-01-15T08:00:00"},
        {"order_id": 2, "customer_id": 11, "amount": 249.50, "status": "shipped",  "note": "gift wrap",   "created_at": "2024-01-15T09:30:00"},
        {"order_id": 3, "customer_id": 12, "amount": 12.00,  "status": "delivered","note": None,          "created_at": "2024-01-15T11:00:00"},
    ]


p = dlt.pipeline(
    pipeline_name="schema_demo",
    destination="duckdb",
    dataset_name="demo",
)

p.run(orders_v1())

with p.sql_client() as client:
    with client.execute_query("SELECT * FROM demo.orders") as cur:
        print(cur.df().to_string(index=False))

**What just happened?**
- dlt used your `columns=` hints instead of inferring types from the data.
- `nullable=False` is a contract hint — dlt will warn/error if NULLs appear in non-nullable columns.
- `created_at` is stored as `timestamp`, not `text`, even though we passed a string — dlt coerces it.

Now inspect the schema to confirm:

In [ ]:
schema = p.default_schema
print(schema.to_pretty_yaml())

---
## Step 3 · Schema evolution — a new field appears

In production, source APIs add new fields all the time. With the default `"evolve"` contract, dlt simply adds the column and continues.

In [ ]:
# Version 2 of orders — a new 'priority' field was added by the source team
@dlt.resource(
    name="orders",
    primary_key="order_id",
    write_disposition="replace",
    # No explicit column hints this time — let dlt infer the new field
)
def orders_v2():
    yield [
        {"order_id": 1, "customer_id": 10, "amount": 99.99,  "status": "pending",   "note": None,        "created_at": "2024-01-15T08:00:00", "priority": "high"},
        {"order_id": 2, "customer_id": 11, "amount": 249.50, "status": "shipped",   "note": "gift wrap", "created_at": "2024-01-15T09:30:00", "priority": "low"},
        {"order_id": 4, "customer_id": 13, "amount": 55.00,  "status": "pending",   "note": None,        "created_at": "2024-01-16T07:00:00", "priority": "medium"},
    ]


# Default schema_contract is "evolve" — new column is added automatically
p2 = dlt.pipeline(
    pipeline_name="schema_evolve",
    destination="duckdb",
    dataset_name="evolve",
)

# Run 1: original schema (no 'priority' column)
p2.run(orders_v1())
schema_after_v1 = list(p2.default_schema.get_table("orders")["columns"].keys())
print("Columns after run 1 (v1 data):", schema_after_v1)

# Run 2: new 'priority' field arrives — schema evolves automatically
p2.run(orders_v2())
schema_after_v2 = list(p2.default_schema.get_table("orders")["columns"].keys())
print("Columns after run 2 (v2 data):", schema_after_v2)
print("\nNew column added:", set(schema_after_v2) - set(schema_after_v1))

In [ ]:
# Print the updated schema as YAML — 'priority' column should now appear
print(p2.default_schema.to_pretty_yaml())

**What just happened?**
- After run 1, the schema had the original 6 columns.
- After run 2, dlt detected the new `priority` field, added it to the schema, and added the column to the DuckDB table.
- Rows from run 1 (which don't have `priority`) now show NULL in that column — this is correct and expected.
- The schema YAML shows `priority` with `data_type: text`.

---
## Step 4 · Schema contract `"freeze"` — block unexpected changes

In [ ]:
from dlt.common.schema.exceptions import SchemaFrozenException

@dlt.resource(
    name="strict_orders",
    primary_key="order_id",
    write_disposition="replace",
    # schema_contract on the resource controls column-level changes
    schema_contract={"columns": "freeze"},
    columns={
        "order_id":    {"data_type": "bigint", "nullable": False, "primary_key": True},
        "customer_id": {"data_type": "bigint", "nullable": False},
        "amount":      {"data_type": "double", "nullable": False},
        "status":      {"data_type": "text",   "nullable": False},
    },
)
def strict_orders_ok():
    """Clean data — matches the declared schema exactly."""
    yield [
        {"order_id": 1, "customer_id": 10, "amount": 99.99,  "status": "pending"},
        {"order_id": 2, "customer_id": 11, "amount": 249.50, "status": "shipped"},
    ]


@dlt.resource(
    name="strict_orders",
    primary_key="order_id",
    write_disposition="replace",
    schema_contract={"columns": "freeze"},
    columns={
        "order_id":    {"data_type": "bigint", "nullable": False, "primary_key": True},
        "customer_id": {"data_type": "bigint", "nullable": False},
        "amount":      {"data_type": "double", "nullable": False},
        "status":      {"data_type": "text",   "nullable": False},
    },
)
def strict_orders_bad():
    """Data with a new 'discount' column — should be blocked by freeze."""
    yield [
        {"order_id": 3, "customer_id": 12, "amount": 12.00, "status": "delivered", "discount": 0.10},
    ]


p_freeze = dlt.pipeline(
    pipeline_name="schema_freeze",
    destination="duckdb",
    dataset_name="frozen",
)

# First run with clean data — should succeed
info_ok = p_freeze.run(strict_orders_ok())
print("Run with clean data: OK")
print(info_ok)

# Second run with an unknown column — should raise SchemaFrozenException
print("\nAttempting run with 'discount' column (contract=freeze)...")
try:
    info_bad = p_freeze.run(strict_orders_bad())
    # If no exception, check if there were load errors
    if info_bad.has_failed_jobs:
        print(f"Load failed (as expected): {info_bad}")
    else:
        print("WARNING: Load succeeded unexpectedly — check your dlt version.")
except (SchemaFrozenException, Exception) as e:
    print(f"Caught expected error: {type(e).__name__}: {str(e)[:200]}")

**What just happened?**
- The first run succeeded because the data exactly matched the declared schema.
- The second run attempted to load a row with `discount` — an undeclared column.
- `schema_contract={"columns": "freeze"}` caused dlt to raise a `SchemaFrozenException` (or mark the job as failed), blocking the load entirely.
- The table is unchanged — no partial loads.

This is critical for production pipelines where an upstream team adding a field could corrupt downstream models.

---
## Step 5 · `discard_value` — load the row, drop the unknown field

In [ ]:
@dlt.resource(
    name="products",
    primary_key="id",
    write_disposition="replace",
    schema_contract={"columns": "discard_value"},  # unknown columns are silently NULLed
    columns={
        "id":    {"data_type": "bigint", "nullable": False, "primary_key": True},
        "name":  {"data_type": "text",   "nullable": False},
        "price": {"data_type": "double", "nullable": False},
    },
)
def products_with_extra():
    """Source sends an extra 'internal_code' field we don't want."""
    yield [
        {"id": 1, "name": "Widget A", "price": 9.99,  "internal_code": "SKU-001"},
        {"id": 2, "name": "Widget B", "price": 14.99, "internal_code": "SKU-002"},
        {"id": 3, "name": "Gadget X", "price": 49.99, "internal_code": "SKU-003"},
    ]


p_discard_val = dlt.pipeline(
    pipeline_name="discard_value_demo",
    destination="duckdb",
    dataset_name="dv",
)

p_discard_val.run(products_with_extra())

with p_discard_val.sql_client() as client:
    with client.execute_query("SELECT * FROM dv.products") as cur:
        df = cur.df()
        print("Loaded table (internal_code should be absent or NULL):")
        print(df.to_string(index=False))
        print(f"\nColumns in table: {list(df.columns)}")
        print("'internal_code' in columns:", "internal_code" in df.columns)

**What just happened?**
- All 3 rows were loaded.
- The `internal_code` values were **silently discarded** — the column was not added to the table.
- The declared columns (`id`, `name`, `price`) are present with correct values.

Use `discard_value` when sources are noisy and you only care about a known subset of fields.

---
## Step 6 · `discard_row` — drop rows that contain unknown fields

In [ ]:
@dlt.resource(
    name="clean_events",
    primary_key="event_id",
    write_disposition="replace",
    schema_contract={"columns": "discard_row"},  # rows with unknown columns are dropped entirely
    columns={
        "event_id": {"data_type": "bigint", "nullable": False, "primary_key": True},
        "type":     {"data_type": "text",   "nullable": False},
        "user_id":  {"data_type": "bigint", "nullable": False},
    },
)
def mixed_events():
    """
    Mix of clean rows and rows with an unexpected 'debug_info' field.
    Rows with 'debug_info' should be dropped.
    """
    yield [
        {"event_id": 1, "type": "click",    "user_id": 42},                                      # clean
        {"event_id": 2, "type": "purchase", "user_id": 7,  "debug_info": "session_x42"},         # has extra field → dropped
        {"event_id": 3, "type": "view",     "user_id": 15},                                      # clean
        {"event_id": 4, "type": "logout",   "user_id": 42, "debug_info": "session_cleanup"},     # has extra field → dropped
        {"event_id": 5, "type": "click",    "user_id": 99},                                      # clean
    ]


p_discard_row = dlt.pipeline(
    pipeline_name="discard_row_demo",
    destination="duckdb",
    dataset_name="dr",
)

p_discard_row.run(mixed_events())

with p_discard_row.sql_client() as client:
    with client.execute_query("SELECT event_id, type, user_id FROM dr.clean_events ORDER BY event_id") as cur:
        df = cur.df()
        print(f"Rows loaded: {len(df)} (expected 3 — events 2 and 4 dropped)")
        print(df.to_string(index=False))

**What just happened?**
- Events 1, 3, 5 (no extra fields) were loaded.
- Events 2 and 4 (containing `debug_info`) were **silently dropped** — not loaded at all.
- Only 3 rows in the destination.

Use `discard_row` when rows with unexpected fields are definitively malformed and should never touch your destination.

---
## Step 7 · All four contract modes side by side

In [ ]:
import dlt
from dlt.common.schema.exceptions import SchemaFrozenException

BASE_COLS = {
    "id":    {"data_type": "bigint", "nullable": False, "primary_key": True},
    "value": {"data_type": "text",   "nullable": False},
}

DATA_WITH_EXTRA = [
    {"id": 1, "value": "alpha", "extra": "X"},
    {"id": 2, "value": "beta"},               # no extra field
]

modes = ["evolve", "discard_value", "discard_row", "freeze"]
results = []

for mode in modes:
    @dlt.resource(
        name="items",
        primary_key="id",
        write_disposition="replace",
        schema_contract={"columns": mode},
        columns=BASE_COLS,
    )
    def items_resource(data=DATA_WITH_EXTRA):
        yield data

    p = dlt.pipeline(
        pipeline_name=f"contract_{mode}",
        destination="duckdb",
        dataset_name="contract_test",
    )

    try:
        p.run(items_resource())
        with p.sql_client() as client:
            with client.execute_query("SELECT COUNT(*) AS n FROM contract_test.items") as cur:
                n = cur.df()["n"][0]
            with client.execute_query("SELECT * FROM contract_test.items") as cur:
                has_extra = "extra" in cur.df().columns
        results.append({"mode": mode, "rows": n, "extra_col_present": has_extra, "error": None})
    except Exception as e:
        results.append({"mode": mode, "rows": 0, "extra_col_present": False, "error": type(e).__name__})

print(f"{'Mode':<16} {'Rows loaded':>11} {'extra col present':>18} {'Error':>25}")
print("-" * 74)
for r in results:
    err = r['error'] or '—'
    print(f"{r['mode']:<16} {r['rows']:>11} {str(r['extra_col_present']):>18} {err:>25}")

Reading the comparison table:
- **evolve**: 2 rows, `extra` column present — schema grew.
- **discard_value**: 2 rows, `extra` column absent — values silently NULLed.
- **discard_row**: 1 row (only id=2, which has no extra field), `extra` column absent.
- **freeze**: 0 rows, raised an error — load blocked entirely.

---
## Step 8 · View the full schema YAML after each change

In [ ]:
# Build a pipeline, run it twice (schema evolves on second run), show YAML both times

@dlt.resource(name="widgets", primary_key="id", write_disposition="replace")
def widgets_v1():
    yield [{"id": 1, "name": "A", "price": 10.00}]

@dlt.resource(name="widgets", primary_key="id", write_disposition="replace")
def widgets_v2():
    # v2 adds 'category' and 'in_stock' fields
    yield [{"id": 1, "name": "A", "price": 10.00, "category": "tools", "in_stock": True}]

p_widgets = dlt.pipeline(
    pipeline_name="schema_yaml_demo",
    destination="duckdb",
    dataset_name="widgets_ds",
)

p_widgets.run(widgets_v1())
print("=== Schema after v1 ===")
schema_v1 = p_widgets.default_schema
cols_v1 = {k: v.get('data_type') for k, v in schema_v1.get_table("widgets")["columns"].items() if not k.startswith('_')}
print(cols_v1)

p_widgets.run(widgets_v2())
print("\n=== Schema after v2 (new columns highlighted) ===")
schema_v2 = p_widgets.default_schema
cols_v2 = {k: v.get('data_type') for k, v in schema_v2.get_table("widgets")["columns"].items() if not k.startswith('_')}
print(cols_v2)

new_cols = set(cols_v2.keys()) - set(cols_v1.keys())
print(f"\nNew columns added by evolution: {new_cols}")

print("\n=== Full schema YAML (v2) ===")
print(schema_v2.to_pretty_yaml())

---
## Step 9 · Setting contracts at the pipeline level

Instead of per-resource contracts, you can set a default contract for the entire pipeline:

In [ ]:
# Pipeline-level schema contract — applies to ALL resources in this pipeline
p_strict = dlt.pipeline(
    pipeline_name="strict_pipeline",
    destination="duckdb",
    dataset_name="strict",
    # schema_contract at pipeline level: controls default for every resource
    # Can be overridden per-resource
)

@dlt.resource(
    name="log_events",
    primary_key="event_id",
    write_disposition="append",
    # Override the pipeline default for this specific resource
    schema_contract={"columns": "evolve", "tables": "freeze"},
    # columns: evolve → allow new columns on existing tables
    # tables: freeze  → block creation of NEW tables entirely
)
def log_events():
    yield [
        {"event_id": 1, "action": "login",  "user_id": 42},
        {"event_id": 2, "action": "logout", "user_id": 42, "duration_sec": 3600},  # new column OK
    ]

info = p_strict.run(log_events())
print("Pipeline with mixed contract (tables=freeze, columns=evolve):")
print(info)

with p_strict.sql_client() as client:
    with client.execute_query("SELECT * FROM strict.log_events ORDER BY event_id") as cur:
        print("\nLoaded rows (duration_sec column auto-added):")
        print(cur.df().to_string(index=False))

---
## Challenge — do this yourself

You're building a production pipeline for customer data. The schema is fixed and must not change without a human sign-off.

1. Define a `customers` resource with these columns: `customer_id` (bigint, PK), `email` (text, not null), `tier` (text, not null), `signup_date` (timestamp, not null). Set `schema_contract={"columns": "freeze"}`.
2. Run the pipeline with 3 clean customer records — should succeed.
3. Create a second version of the resource that includes an extra `referral_code` field.
4. Try to run the pipeline with the second version — catch the error and print a friendly message like `"Schema violation detected: contact the data team."`.
5. Bonus: print the schema YAML after the successful first run to confirm only 4 columns are declared.

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
import dlt
from dlt.common.schema.exceptions import SchemaFrozenException

CUSTOMER_SCHEMA = {
    "customer_id":  {"data_type": "bigint",    "nullable": False, "primary_key": True},
    "email":        {"data_type": "text",      "nullable": False},
    "tier":         {"data_type": "text",      "nullable": False},
    "signup_date":  {"data_type": "timestamp", "nullable": False},
}

@dlt.resource(
    name="customers",
    primary_key="customer_id",
    write_disposition="replace",
    schema_contract={"columns": "freeze"},
    columns=CUSTOMER_SCHEMA,
)
def customers_clean():
    yield [
        {"customer_id": 1, "email": "alice@example.com", "tier": "gold",   "signup_date": "2024-01-01T00:00:00"},
        {"customer_id": 2, "email": "bob@example.com",   "tier": "silver", "signup_date": "2024-01-05T00:00:00"},
        {"customer_id": 3, "email": "carol@example.com", "tier": "gold",   "signup_date": "2024-01-10T00:00:00"},
    ]

@dlt.resource(
    name="customers",
    primary_key="customer_id",
    write_disposition="replace",
    schema_contract={"columns": "freeze"},
    columns=CUSTOMER_SCHEMA,
)
def customers_with_referral():
    yield [
        {"customer_id": 4, "email": "dave@example.com", "tier": "bronze",
         "signup_date": "2024-01-15T00:00:00", "referral_code": "REF-XYZ"},
    ]

p = dlt.pipeline(pipeline_name="customer_challenge", destination="duckdb", dataset_name="customers")

# Step 2: clean run
p.run(customers_clean())
print("Clean run: OK")

# Bonus: print schema
cols = {k: v.get('data_type') for k, v in p.default_schema.get_table('customers')['columns'].items() if not k.startswith('_')}
print(f"Declared columns: {cols}")

# Step 4: contract violation
try:
    p.run(customers_with_referral())
except Exception:
    print("Schema violation detected: contact the data team.")
```
</details>

---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `columns=` hint | Declare explicit types and nullability — don't rely solely on inference in production |
| `data_type` options | `text`, `bigint`, `double`, `bool`, `date`, `timestamp`, `json`, `binary` |
| Schema evolution | Default behaviour — new fields are auto-added as new columns |
| `"evolve"` contract | The default; schema grows automatically with source changes |
| `"freeze"` contract | Blocks any schema change — raises `SchemaFrozenException` |
| `"discard_value"` contract | Loads the row; silently NULLs the unknown column value |
| `"discard_row"` contract | Drops any row that contains an unknown column |
| `schema.to_pretty_yaml()` | Human-readable snapshot of the full schema — great for auditing |
| `schema.get_table(name)` | Programmatic access to column definitions for a single table |
| Contract scope | Apply to `tables`, `columns`, and/or `data_type` independently |

> **Tip:** Schema freezing is critical for production pipelines. Set `schema_contract={"columns": "freeze"}` on any resource whose schema is stable, and treat any `SchemaFrozenException` as a deployment gate — not something to auto-fix.

---
## What's next

You have completed all 5 days of the **dlt Certified** course.

You now know how to:
- Build custom sources with `@dlt.resource` and `@dlt.source`
- Call REST APIs using the `rest_api` helper with pagination and auth
- Choose the right write disposition for every scenario
- Load incrementally using cursor fields and dlt state
- Declare explicit schemas and enforce data contracts in production

Mark Day 5 complete in your [tracker](../index.html).